In [0]:
# Simulamos un entorno productivo limpio
%pip install mlflow scikit-learn pandas --quiet
dbutils.library.restartPython()

In [0]:

import sys, os
import mlflow
import pandas as pd
import numpy as np
from pyspark.sql.functions import current_timestamp

notebook_path = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_path, ".."))
FOLDER_NAME = "src" 

src_path = os.path.join(project_root, FOLDER_NAME)
if not os.path.exists(src_path): src_path = os.path.join(notebook_path, FOLDER_NAME)
if src_path not in sys.path: sys.path.append(src_path)

In [0]:

from nombre_paquete.preprocessing import transformers
print(" Entorno de despliegue configurado.")

model_name = "Climate_Energy_Predictor"
stage = "Production"

print(f" Conectando con Model Registry...")
print(f"   Buscando modelo: {model_name} (Etapa: {stage})")

try:
    # Carga dinámica
    model_uri = f"models:/{model_name}@{stage.lower()}"
    production_model = mlflow.sklearn.load_model(model_uri)
    print(" Modelo cargado exitosamente y listo para inferencia.")
except Exception as e:
    print(f" Error crítico: No se encontró un modelo en estado '{stage}'.")
    print("  Asegúrate de haber ejecutado el Notebook 05 (Evaluation) y aprobado el modelo.")
    raise e


In [0]:

# --- CELDA 4: SIMULACIÓN DE DATOS NUEVOS (INPUT) ---
# En la vida real, aquí harías: df_new = spark.table("bronze_new_data").toPandas()
# Para este ejemplo, generamos datos futuros sintéticos.

print(" Recibiendo datos de la próxima semana...")

dates_future = pd.date_range(start='2024-02-01', periods=7, freq='D') # 7 días a futuro
countries = ['Germany', 'France', 'Spain'] 

# Creamos un DataFrame con la estructura cruda
new_data_raw = pd.DataFrame({
    'date': dates_future.repeat(len(countries)),
    'country': countries * len(dates_future),
    # Valores simulados (Ej: se prevé una semana fría)
    'avg_temperature': np.random.uniform(5, 12, size=len(dates_future)*len(countries)), 
    'humidity': np.random.uniform(50, 80, size=len(dates_future)*len(countries)),
    'co2_emission': np.random.uniform(150, 250, size=len(dates_future)*len(countries)),
    'renewable_share': [35] * len(dates_future) * len(countries),
    'urban_population': [800000] * len(dates_future) * len(countries), # Dato estático
    'industrial_activity_index': [55] * len(dates_future) * len(countries),
    'energy_price': [0.18] * len(dates_future) * len(countries)
})

print(f" Datos recibidos: {new_data_raw.shape[0]} filas para procesar.")
display(new_data_raw.head(3))


In [0]:

# --- CELDA 5: PREPROCESAMIENTO (Batch Processing) ---
print(" Aplicando pipeline de limpieza (Transformers)...")

# 1. Limpieza básica y Feature Engineering (Fechas, Lags, etc.)
df_clean = transformers.preprocess_data(new_data_raw)

# 2. Alineación de columnas (CRÍTICO EN PRODUCCIÓN)
# El modelo espera ver EXACTAMENTE las mismas columnas que cuando se entrenó.
# A veces, al procesar pocos datos, faltan columnas (ej: si no aparece 'Spain', 
# no se crea la columna 'country_Spain'). Aquí lo arreglamos.

# Obtenemos las columnas que el modelo espera (si tiene el atributo)
if hasattr(production_model, "feature_names_in_"):
    expected_cols = production_model.feature_names_in_
else:
    # Si no, asumimos que df_clean ya viene bien (riesgoso pero funcional para demo)
    expected_cols = df_clean.columns

# Rellenamos con 0 las columnas faltantes (ej: países que no están en la data nueva)
for col in expected_cols:
    if col not in df_clean.columns:
        df_clean[col] = 0

# Ordenamos las columnas igual que en el entrenamiento
df_final_input = df_clean[expected_cols]

print(" Datos listos para el modelo.")


In [0]:

# --- CELDA 6: INFERENCIA (PREDICCIÓN) ---
print(" Ejecutando predicción...")

try:
    predictions = production_model.predict(df_final_input)
    
    # Pegamos la predicción al lado de la fecha y país para que sea legible
    # Usamos el índice de df_clean que tiene la fecha
    results_df = df_clean.copy()
    results_df = results_df.reset_index() # Recuperar fecha como columna
    
    # Recuperamos el país (truco: revertir one-hot o usar el raw original si coinciden índices)
    # Para simplificar, usaremos las fechas y pegaremos el valor
    results_df['PRED_ENERGY_CONSUMPTION'] = predictions
    
    # Seleccionamos solo lo importante para el reporte
    output_cols = ['date', 'PRED_ENERGY_CONSUMPTION']
    # Intentamos recuperar columnas de países si existen
    country_cols = [c for c in results_df.columns if 'country_' in c]
    output_cols.extend(country_cols)
    
    final_report = results_df[output_cols]
    
    print(" Predicciones generadas exitosamente.")
    display(final_report.head())

except Exception as e:
    print(f" Error en la predicción: {e}")
    print("Tip: Revisa si escalaste los datos. En un entorno 100% real, debes cargar también el 'scaler' usado en train.")


In [0]:

# --- CELDA 7: GUARDAR EN CAPA GOLD (Resultados Finales) ---
print(" Guardando resultados en tabla 'Gold' para PowerBI...")

# Limpieza de nombres para Delta (sin espacios ni paréntesis)
final_report.columns = [c.replace(' ', '_').replace('(', '').replace(')', '') for c in final_report.columns]

# Convertir a Spark DataFrame
spark_df = spark.createDataFrame(final_report)

# Añadimos fecha de ejecución (metadata para auditoría)
spark_df = spark_df.withColumn("execution_timestamp", current_timestamp())

# Guardar en Delta Lake (Mode Append: agregar al histórico)
table_name = "climate_predictions_gold"
spark_df.write.format("delta").mode("append").saveAsTable(table_name)

print(f" ¡Ciclo completado! Los datos están disponibles en la tabla '{table_name}'.")
print("   Ahora puedes conectar Tableau o PowerBI a esta tabla.")